# CS425 Diss-Sonnet Generator — Pretrained Small LLM

This notebook adapts the pretrained `distilgpt2` causal language model with QLoRA. It follows the two-stage design of `Train.ipynb`: sonnet style training, then trait-conditioned diss specialization. A pronunciation dictionary guides 14-line generation toward **ABAB CDCD EFEF GG** rhyme and a configurable syllable target.

Run top to bottom in a Google Colab GPU runtime. Put `sonnets.txt` and `diss.csv` in `My Drive/CS425/Datasets`, or use the upload cell. The optional Hugging Face sonnet source contains versions of the local Shakespeare poems, so the notebook uses one source at a time to prevent near-duplicate poems crossing the validation split. The notebook reports validation loss for both stages and phonetic metrics for the generated example.

Most diss rows are couplets, so the model does not see complete diss sonnets during fine-tuning. The generation constraints provide the sonnet form and may use a line from the diss dataset when model sampling fails; human review is still needed for originality, meter, coherence, and diss quality.


## Part 0 — Install QLoRA dependencies

This cell installs `transformers`, `datasets`, `peft`, `bitsandbytes`, and `accelerate`. QLoRA requires a CUDA-capable runtime and a fresh kernel after installation.

In [ ]:
"""
PRETRAINED MODEL NOTEBOOK — PART 0: DEPENDENCY INSTALLATION

Install the libraries required for 4-bit QLoRA fine-tuning.
"""

import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "nltk": "nltk",
    "datasets": "datasets",
    "transformers": "transformers",
    "peft": "peft",
    "bitsandbytes": "bitsandbytes",
    "accelerate": "accelerate",
}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("QLoRA dependencies have been installed.")
else:
    print("QLoRA dependencies are already installed.")

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is missing. Select a CUDA-capable runtime for QLoRA.")



## Part 1 — Imports, reproducibility, and GPU check

### What this block does
Imports libraries, fixes random seeds, checks whether CUDA is available, mounts Google Drive in Colab, and creates the working directories.

### Methodology
Datasets and cleaned copies live in `My Drive/CS425/Datasets`; checkpoints and plots stay in the Colab runtime until the final archive is downloaded. A fixed seed makes model initialization and dataset splitting more reproducible. Exact equality between GPU runs is not guaranteed, but this significantly reduces accidental variation.


In [ ]:
"""
PRETRAINED MODEL NOTEBOOK — PART 1: ENVIRONMENT SETUP
"""

from pathlib import Path
import json
import math
import random
import re
import os

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import accelerate
import bitsandbytes
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROJECT_DIR = (
    Path("/content/diss_poem_project") if Path("/content").is_dir()
    else Path.cwd() / "diss_poem_project"
)

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    drive = None
    IS_COLAB = False

if IS_COLAB:
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/CS425/Datasets")
else:
    DATA_DIR = Path.cwd() / "Datasets"

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
OUTPUT_DIR = PROJECT_DIR / "outputs"
for directory in [DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Dataset folder:", DATA_DIR)
print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)


## Part 2 — Configuration

The raw diss CSV takes priority when it is present; a cleaned CSV is used only if the raw file is unavailable. This prevents old cleaned copies from hiding newly added training rows. `positive_traits` conditions the diss stage. `SONNET_DATASET_NAME` may be set to `None` to train only on the local sonnets.


In [ ]:
"""Paths, model and training settings for the pretrained baseline."""
SONNET_PATH = DATA_DIR / "sonnets.txt"
DISS_PATH = DATA_DIR / "diss.csv"
PRON_DICT_PATH = DATA_DIR / "csudict.dict"
CLEANED_DIR = DATA_DIR / "cleaned"
SONNET_CLEANED_PATH = CLEANED_DIR / "sonnets_cleaned.txt"
DISS_CLEANED_PATH = CLEANED_DIR / "diss_cleaned.csv"
FORCE_UPLOAD = False
DISS_TEXT_COLUMN = "text"
DISS_POSITIVE_TRAITS_COLUMN = "positive_traits"
USE_NLTK_CMUDICT_IF_MISSING = True
LOWERCASE = True
VALIDATION_SPLIT = 0.10

BASE_MODEL_NAME = "distilgpt2"
SONNET_DATASET_NAME = "zhyncs/sonnet"  # Set to None to use only local sonnets.
MAX_SEQUENCE_LENGTH = 512
QLORA_R = 16
QLORA_ALPHA = 32
QLORA_DROPOUT = 0.05
SONNET_LEARNING_RATE = 2e-4
DISS_LEARNING_RATE = 1e-4
SONNET_EPOCHS = 2
DISS_EPOCHS = 2
QLORA_BATCH_SIZE = 2
QLORA_GRADIENT_ACCUMULATION = 8
QLORA_OUTPUT_DIR = CHECKPOINT_DIR / "qlora_distilgpt2"

MAX_CONTEXT_TOKENS = 768
MAX_NEW_TOKENS_PER_LINE = 40
GENERATION_TEMPERATURE = 0.85
TOP_K = 40
TOP_P = 0.92
CANDIDATES_PER_LINE = 8
MAX_GENERATION_ROUNDS = 3
ALLOW_CORPUS_FALLBACK = False  # Set False to evaluate model-generated lines only.
RHYME_WEIGHT = 2.5
REPEAT_PENALTY = 3.0
LEXICON_PENALTY = 1.5
MAX_UNKNOWN_WORD_FRACTION = 0.25
TARGET_SYLLABLES = 10
METER_TOLERANCE = 0  # Require exactly 10 estimated syllables per line.

print("Base model:", BASE_MODEL_NAME)
print("Optional extra sonnets:", SONNET_DATASET_NAME)


## Part 3 — Locate datasets

The notebook uses files in `My Drive/CS425/Datasets` on Colab or the repository `Datasets/` folder locally. The upload cell copies available files or prompts when a required corpus is missing. A cleaned sonnet file is preferred when present because it preserves poem boundaries in a raw source without blank lines. The raw diss CSV is preferred so added rows are not hidden by an older cleaned copy. Set `FORCE_UPLOAD = True` to replace a Drive file.


In [ ]:
"""
TRAINING PROCESS — PART 3: DATASET UPLOAD

Prompt only when a required corpus is missing. Accept raw and _cleaned files.
"""

import shutil

# Use files already present in the runtime, including a cloned project folder.
candidate_dirs = [
    Path.cwd() / "Datasets",
    Path.cwd() / "CS425-Project" / "Datasets",
    Path.cwd(),
]


def copy_available(relative_name, destination):
    for directory in candidate_dirs:
        source = directory / relative_name
        if source.is_file():
            if source.resolve() != destination.resolve():
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(source, destination)
            print(f"Found {source} -> {destination}")
            return True
    return False


if not (SONNET_CLEANED_PATH.exists() or SONNET_PATH.exists()):
    copy_available("cleaned/sonnets_cleaned.txt", SONNET_CLEANED_PATH)
    if not SONNET_CLEANED_PATH.exists():
        copy_available("Sonnet-154.txt", SONNET_PATH) or copy_available("sonnets.txt", SONNET_PATH)
if not (DISS_CLEANED_PATH.exists() or DISS_PATH.exists()):
    copy_available("cleaned/diss_cleaned.csv", DISS_CLEANED_PATH)
    if not DISS_CLEANED_PATH.exists():
        copy_available("diss.csv", DISS_PATH)
if not PRON_DICT_PATH.exists():
    copy_available("csudict.dict", PRON_DICT_PATH) or copy_available("cmudict.dict", PRON_DICT_PATH)

missing = []
if not (SONNET_PATH.exists() or SONNET_CLEANED_PATH.exists()):
    missing.append("Sonnet-154.txt or sonnets_cleaned.txt")
if not (DISS_PATH.exists() or DISS_CLEANED_PATH.exists()):
    missing.append("diss.csv or diss_cleaned.csv")

if missing or FORCE_UPLOAD:
    if not IS_COLAB:
        print("Running locally. Place the missing datasets in this folder:")
        print(" ", DATA_DIR)
        for name in missing:
            print("   -", name)
        print("Then re-run this cell. (Interactive upload is a Colab feature.)")
        raise FileNotFoundError(
            f"Missing datasets: {', '.join(missing)}. "
            f"Put them in {DATA_DIR} and re-run."
        )

    if missing:
        print("Upload these missing datasets:", ", ".join(missing))
    else:
        print("Optional upload requested; choose files to add or replace.")
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Place the missing datasets in the configured data folder "
            "or run this cell in Google Colab to upload them."
        ) from exc

    uploaded = files.upload()

    for filename, data in uploaded.items():
        source_name = Path(filename).name.lower()

        if source_name.endswith(".txt") and "sonnet" in source_name:
            destination = (
                SONNET_CLEANED_PATH if source_name.endswith("_cleaned.txt")
                else SONNET_PATH
            )
        elif source_name.endswith(".csv") and "diss" in source_name:
            destination = (
                DISS_CLEANED_PATH if source_name.endswith("_cleaned.csv")
                else DISS_PATH
            )
        elif (
            source_name.endswith(".dict")
            or "cmudict" in source_name
            or "csudict" in source_name
        ):
            destination = PRON_DICT_PATH
        else:
            destination = DATA_DIR / Path(filename).name

        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)
        print(f"Saved {filename} -> {destination}")
else:
    print("Sonnet and diss datasets found; skipping upload.")


## Part 4 — Load and clean the datasets

Load local sonnets and diss rows. The cleaned local sonnet file preserves poem boundaries; the raw diss CSV takes priority to include recent rows. If the optional Hugging Face source is available, use its complete 14-line poems in place of the local Shakespeare poems, since they overlap. Normalization preserves line breaks. Diss examples keep `positive_traits` as the prompt and diss text as the completion. Complete samples are split into training and validation sets before tokenization.


In [ ]:
"""Load current local data; optionally enrich sonnets from Hugging Face."""
def clean_text(text: str, lowercase: bool = True) -> str:
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    lines = [re.sub(r"\s+", " ", line).strip() for line in text.split("\n")]
    text = "\n".join(line for line in lines if line)
    return text.lower() if lowercase else text


def split_sonnet_samples(raw_text: str):
    samples = [sample for sample in re.split(r"\n\s*\n", raw_text) if sample.strip()]
    if len(samples) == 1:
        lines = [line for line in raw_text.splitlines() if line.strip()]
        if len(lines) >= 28:
            if len(lines) % 14:
                print("WARNING: local sonnet line count is not divisible by 14.")
            return ["\n".join(lines[i:i + 14]) for i in range(0, len(lines), 14)]
    return samples


def load_sonnet_txt(path: Path, cleaned_path: Path):
    source = cleaned_path if cleaned_path.exists() else path
    if not source.exists():
        raise FileNotFoundError(f"Sonnet file not found: {path} or {cleaned_path}")
    samples = split_sonnet_samples(source.read_text(encoding="utf-8-sig", errors="ignore"))
    samples = [clean_text(sample, LOWERCASE) for sample in samples]
    return [sample for sample in samples if len(sample.splitlines()) == 14]


def load_hf_sonnets(dataset_name):
    if not dataset_name:
        return []
    from datasets import load_dataset
    try:
        hf = load_dataset(dataset_name, split="train")
    except (OSError, ConnectionError) as exc:
        print(f"Could not load optional {dataset_name}: {exc}")
        return []
    rows = [clean_text(row.get("text", ""), LOWERCASE) for row in hf]
    if all(len(row.splitlines()) <= 1 for row in rows):
        # zhyncs/sonnet stores Shakespeare's sonnets as line rows. It also
        # contains one roman-numeral heading, a 15-line sonnet (99), and a
        # 12-line sonnet (126). Remove the heading before grouping by poem.
        lines = [row for row in rows if row and not re.fullmatch(r"[ivxlcdm]+\.", row)]
        if dataset_name == "zhyncs/sonnet" and len(lines) == 2155:
            lengths = [14] * 98 + [15] + [14] * 26 + [12] + [14] * 28
        elif len(lines) % 14 == 0:
            lengths = [14] * (len(lines) // 14)
        else:
            raise ValueError(
                f"Cannot infer poem boundaries in {dataset_name} ({len(lines)} lines). "
                "Set SONNET_DATASET_NAME=None to use the local sonnets."
            )
        samples = []
        offset = 0
        for length in lengths:
            if length == 14:
                samples.append("\n".join(lines[offset:offset + length]))
            offset += length
        assert offset == len(lines)
    else:
        samples = [row for row in rows if len(row.splitlines()) == 14]
    print(f"Grouped {len(rows):,} dataset rows into {len(samples):,} complete 14-line sonnets.")
    return samples


def load_diss_csv(path: Path, text_column: str, positive_column: str, cleaned_path: Path):
    source = path if path.exists() else cleaned_path
    if not source.exists():
        raise FileNotFoundError(f"Diss CSV not found: {path} or {cleaned_path}")
    df = pd.read_csv(source, encoding="utf-8-sig")
    required = {text_column, positive_column}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns {sorted(missing)} in {source}")
    df = df.dropna(subset=[text_column, positive_column])
    samples = []
    for text, traits in zip(df[text_column], df[positive_column]):
        text = clean_text(text, LOWERCASE)
        traits = clean_text(traits, LOWERCASE).replace("\n", " ")
        if text and traits:
            samples.append({"text": text, "positive_traits": traits})
    print(f"Diss source: {source} ({len(samples):,} usable rows)")
    return samples


local_sonnet_samples = load_sonnet_txt(SONNET_PATH, SONNET_CLEANED_PATH)
extra_sonnet_samples = load_hf_sonnets(SONNET_DATASET_NAME)
# The two sources contain versions of the same Shakespeare poems; using both
# would put near-duplicates on opposite sides of the validation split.
sonnet_samples = extra_sonnet_samples if extra_sonnet_samples else local_sonnet_samples
diss_samples = load_diss_csv(
    DISS_PATH, DISS_TEXT_COLUMN, DISS_POSITIVE_TRAITS_COLUMN, DISS_CLEANED_PATH
)
if len(sonnet_samples) < 2 or len(diss_samples) < 2:
    raise ValueError("Both stages need at least two complete samples.")
print(f"Local sonnets: {len(local_sonnet_samples):,}")
print(f"Extra sonnets: {len(extra_sonnet_samples):,}")
print(f"Training sonnets: {len(sonnet_samples):,}")
print(f"Diss samples: {len(diss_samples):,}")


## Two-stage QLoRA training

The pretrained base stays frozen in 4-bit precision. Stage 1 trains LoRA adapters on complete sonnets. Stage 2 continues those adapters on trait-conditioned diss examples. Each stage splits whole samples before tokenization and reports validation loss and perplexity. The best checkpoint for each stage is restored before continuing.


In [ ]:
"""Load a small pretrained causal LM in 4-bit and attach LoRA adapters."""
if not torch.cuda.is_available():
    raise RuntimeError("QLoRA requires a CUDA GPU. Select a Colab GPU runtime.")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)
hf_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if hf_tokenizer.pad_token is None:
    hf_tokenizer.pad_token = hf_tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, quantization_config=quantization_config, device_map="auto"
)
model.config.pad_token_id = hf_tokenizer.pad_token_id
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=QLORA_R, lora_alpha=QLORA_ALPHA, lora_dropout=QLORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM", target_modules=["c_attn", "c_proj"],
))
model.print_trainable_parameters()


## Stage 2 — Trait-conditioned diss specialization

Most diss samples have two lines. This stage teaches diss vocabulary and trait conditioning, while stage 1 supplies sonnet examples. It does not by itself teach complete 14-line diss poems. The inference cell enforces the 14-line form and rhyme scheme.


In [ ]:
"""Train and validate the same LoRA adapter in two consecutive stages."""
from datasets import Dataset


def make_sonnet_training_text(sample):
    return f"Sonnet:\n{sample}\n{hf_tokenizer.eos_token}"


def make_diss_training_text(pair):
    return (
        f"Subject traits: {pair['positive_traits']}\n"
        "Diss lines:\n"
        f"{pair['text']}\n{hf_tokenizer.eos_token}"
    )


def split_samples(samples, seed):
    shuffled = list(samples)
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, min(len(shuffled) - 1, round(len(shuffled) * VALIDATION_SPLIT)))
    return shuffled[n_val:], shuffled[:n_val]


def tokenize_training_examples(texts):
    dataset = Dataset.from_dict({"text": texts})
    return dataset.map(
        lambda batch: hf_tokenizer(batch["text"], truncation=True, max_length=MAX_SEQUENCE_LENGTH),
        batched=True, remove_columns=["text"],
    )


def causal_lm_collator(features):
    """Mask only padding; GPT-2 uses EOS as PAD, so EOS remains a training target."""
    batch = hf_tokenizer.pad(features, padding=True, return_tensors="pt")
    batch["labels"] = batch["input_ids"].clone()
    batch["labels"][batch["attention_mask"] == 0] = -100
    return batch


def run_qlora_stage(samples, stage_name, epochs, learning_rate, seed):
    train_samples, val_samples = split_samples(samples, seed)
    train_data = tokenize_training_examples(train_samples)
    val_data = tokenize_training_examples(val_samples)
    stage_dir = QLORA_OUTPUT_DIR / stage_name
    steps_per_epoch = math.ceil(
        len(train_data) / (QLORA_BATCH_SIZE * QLORA_GRADIENT_ACCUMULATION)
    )
    warmup_steps = max(1, round(steps_per_epoch * epochs * 0.05))
    args = TrainingArguments(
        output_dir=str(stage_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=QLORA_BATCH_SIZE,
        per_device_eval_batch_size=QLORA_BATCH_SIZE,
        gradient_accumulation_steps=QLORA_GRADIENT_ACCUMULATION,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        logging_steps=25,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        optim="paged_adamw_8bit",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        report_to="none",
        remove_unused_columns=False,
        seed=seed,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_data, eval_dataset=val_data,
        data_collator=causal_lm_collator,
    )
    trainer.train()
    metrics = trainer.evaluate()
    loss = metrics["eval_loss"]
    print(f"{stage_name}: train={len(train_samples)}, validation={len(val_samples)}, "
          f"validation loss={loss:.3f}, perplexity={math.exp(min(loss, 50)):.2f}")
    return trainer, metrics


sonnet_texts = [make_sonnet_training_text(s) for s in sonnet_samples]
diss_texts = [make_diss_training_text(p) for p in diss_samples]
sonnet_trainer, sonnet_metrics = run_qlora_stage(
    sonnet_texts, "sonnet", SONNET_EPOCHS, SONNET_LEARNING_RATE, SEED
)
sonnet_adapter_path = QLORA_OUTPUT_DIR / "sonnet_adapter"
model.save_pretrained(sonnet_adapter_path)
diss_trainer, diss_metrics = run_qlora_stage(
    diss_texts, "diss", DISS_EPOCHS, DISS_LEARNING_RATE, SEED + 1
)
FINAL_MODEL_PATH = QLORA_OUTPUT_DIR / "final_adapter"
model.save_pretrained(FINAL_MODEL_PATH)
hf_tokenizer.save_pretrained(FINAL_MODEL_PATH)
model.config.use_cache = True
print("Saved final adapter and tokenizer to:", FINAL_MODEL_PATH)



# Stage 3 — Pronunciation-based rhyme and meter control

## Part 12 — Load CSUdict / CMUdict pronunciation data

### What this block does
Loads a custom CMU-style pronunciation dictionary if provided. Otherwise, it downloads NLTK's CMUdict as a fallback.

### Methodology
The pronunciation dictionary is **not treated as language-model training text**. It acts as a structured phonetic knowledge base after training.

It provides:

```text
word → phonemes → rhyme ending + syllable estimate
```

This keeps the responsibilities separate:
- neural model → content and style
- pronunciation dictionary → phonetic constraints


In [ ]:

"""
TRAINING PROCESS — PART 12: PRONUNCIATION DICTIONARY

Purpose:
    Load phoneme sequences for rhyme and syllable analysis.

Methodology:
    Prefer the project's custom CSUdict/CMUdict-style file.
    If it is missing and fallback is enabled, download NLTK CMUdict.
"""

def load_custom_pronunciation_dictionary(path: Path):
    """
    Parse CMU-style dictionary lines:
        WORD  PHONEME PHONEME ...
    """
    pronunciations = {}

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith(";;;"):
                continue

            parts = line.split()

            if len(parts) < 2:
                continue

            raw_word = parts[0]
            word = re.sub(r"\(\d+\)$", "", raw_word).lower()
            phones = parts[1:]

            pronunciations.setdefault(word, []).append(phones)

    return pronunciations


if PRON_DICT_PATH.exists():
    pronunciations = load_custom_pronunciation_dictionary(PRON_DICT_PATH)
    pronunciation_source = str(PRON_DICT_PATH)

elif USE_NLTK_CMUDICT_IF_MISSING:
    import nltk

    try:
        nltk.data.find("corpora/cmudict")
    except LookupError:
        if not nltk.download("cmudict", quiet=True):
            raise RuntimeError("Could not download NLTK CMUdict.")
    from nltk.corpus import cmudict

    pronunciations = cmudict.dict()
    pronunciation_source = "NLTK CMUdict fallback"

else:
    raise FileNotFoundError(
        f"No pronunciation dictionary found at {PRON_DICT_PATH}"
    )

print("Pronunciation source:", pronunciation_source)
print("Words with pronunciations:", len(pronunciations))



## Part 13 — Convert pronunciations into rhyme and syllable features

### What this block does
Defines functions to:
- extract rhyme keys from ARPAbet-like phoneme sequences
- test whether two words rhyme
- estimate syllables in a word
- estimate syllables in a generated line

### Methodology
A rhyme key is taken from the **last stressed vowel onward**. Two words receive a rhyme match if any known pronunciation shares the same rhyme key.

Syllables are approximated by counting phonemes marked with ARPAbet stress digits (`0`, `1`, or `2`).


In [ ]:
"""
TRAINING PROCESS — PART 13: PHONETIC FEATURE EXTRACTION

Purpose:
    Produce rhyme keys and syllable counts from pronunciation data.

Methodology:
    - Rhyme key: final stressed vowel + following phonemes
    - Syllables: count phonemes carrying stress digits
"""

def pronunciations_for(word):
    """
    Return all known pronunciations for a normalized word.
    """
    if not word:
        return []

    return pronunciations.get(word.lower(), [])


def rhyme_key_from_phones(phones):
    """
    Extract the final stressed vowel and everything after it.
    """
    start_index = None

    # First preference: final primary/secondary stressed vowel.
    for index in range(len(phones) - 1, -1, -1):
        if re.search(r"[12]$", phones[index]):
            start_index = index
            break

    # A weak, unstressed vowel is not enough for a sonnet end rhyme.
    # For example CMUdict has an unstressed "them" pronunciation that
    # would incorrectly match stressed "sum" if stress were discarded.
    if start_index is None:
        return None

    return tuple(
        re.sub(r"\d", "", phone)
        for phone in phones[start_index:]
    )


def rhyme_keys(word):
    """
    Return all unique rhyme keys for all known pronunciations.
    """
    keys = set()

    for phones in pronunciations_for(word):
        key = rhyme_key_from_phones(phones)

        if key is not None:
            keys.add(key)

    return keys


def rhyme_score(word_a, word_b, allow_identical=False):
    """
    Binary rhyme score:
        1.0 = at least one pronunciation rhyme matches
        0.0 = no known match
    """
    if not word_a or not word_b:
        return 0.0

    word_a = word_a.lower()
    word_b = word_b.lower()

    if not allow_identical and word_a == word_b:
        return 0.0

    keys_a = rhyme_keys(word_a)
    keys_b = rhyme_keys(word_b)

    if not keys_a or not keys_b:
        return 0.0

    return float(bool(keys_a.intersection(keys_b)))


def syllable_count_word(word):
    """
    Estimate syllables from the first available pronunciation.
    """
    entries = pronunciations_for(word)

    if not entries:
        cleaned = re.sub(r"[^a-z]", "", word.lower())
        if not cleaned:
            return 0
        cleaned = re.sub(r"[^aeiou]e$", "", cleaned)
        return max(1, len(re.findall(r"[aeiouy]+", cleaned)))

    phones = entries[0]

    return sum(
        bool(re.search(r"[012]$", phone))
        for phone in phones
    )


def words_in_line(line):
    """
    Extract word tokens for phonetic analysis.
    """
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", line)


def last_word(line):
    """
    Return the final word in a generated line.
    """
    words = words_in_line(line)

    if not words:
        return None

    return words[-1].lower()


def syllable_count_line(line):
    """
    Sum pronunciation counts, with a spelling estimate for unknown words.
    """
    return sum(
        syllable_count_word(word)
        for word in words_in_line(line)
    )


def known_word(word):
    """
    True if the word (letters only, lowercased) is in the pronunciation
    dictionary. Used as a lightweight "is this a real English word" check.
    """
    word = re.sub(r"[^a-z]", "", word.lower())
    return bool(word) and word in pronunciations

def unknown_word_fraction(line):
    """
    Fraction of a line's words that the lexicon does not know.
    0.0 = every word is a known word; higher = more likely to be
    a garbled / invented sequence of letters.
    """
    words = words_in_line(line)
    if not words:
        return 1.0
    unknown = sum(0 if known_word(word) else 1 for word in words)
    return unknown / len(words)


# Quick sanity check.
print("Rhyme keys for 'cat':", rhyme_keys("cat"))
print("Rhyme keys for 'hat':", rhyme_keys("hat"))
print("cat / hat rhyme score:", rhyme_score("cat", "hat"))
print("Syllables in 'attack':", syllable_count_word("attack"))


# Inference — Generate a diss sonnet

The model proposes lines from the same trait prompt used in training. Candidates are scored for likelihood, pronunciation-based rhyme and syllables, word quality, and repetition. A rhyme fallback tests phonetic replacements for the final word when sampling produces no matching ending.


In [ ]:
"""Sample one line; score complete lines with a separate teacher-forced pass."""
def first_generated_line(continuation):
    """Skip leading blank lines or echoed prompt labels in model output."""
    for raw_line in continuation.splitlines():
        line = raw_line.strip()
        if line and not line.lower().startswith(("subject traits:", "diss lines:", "sonnet:")):
            return line
    return ""


@torch.no_grad()
def generate_candidate_line(model, prompt, temperature=GENERATION_TEMPERATURE):
    model.eval()
    encoded = hf_tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    # Keep the latest lines when the poem exceeds the model context window.
    input_ids = encoded["input_ids"][:, -MAX_CONTEXT_TOKENS:].to(DEVICE)
    attention_mask = encoded["attention_mask"][:, -MAX_CONTEXT_TOKENS:].to(DEVICE)
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS_PER_LINE,
        min_new_tokens=6,
        do_sample=True,
        temperature=temperature,
        top_k=TOP_K,
        top_p=TOP_P,
        pad_token_id=hf_tokenizer.pad_token_id,
        eos_token_id=hf_tokenizer.eos_token_id,
        repetition_penalty=1.12,
    )[0]
    continuation = hf_tokenizer.decode(
        output[input_ids.shape[1]:], skip_special_tokens=True
    )
    return first_generated_line(continuation)


@torch.no_grad()
def line_log_likelihood(model, prompt, line):
    """Mean log probability of line tokens under the adapted language model."""
    model.eval()
    prefix_length = len(hf_tokenizer(prompt, add_special_tokens=False)["input_ids"])
    full_ids = hf_tokenizer(prompt + line, add_special_tokens=False)["input_ids"]
    cut = max(0, len(full_ids) - MAX_CONTEXT_TOKENS)
    ids = torch.tensor([full_ids[cut:]], device=DEVICE)
    first_target = max(1, prefix_length - cut)
    if first_target >= ids.shape[1]:
        return float("-inf")
    logits = model(input_ids=ids).logits[0, first_target - 1:-1]
    targets = ids[0, first_target:]
    token_log_probs = F.log_softmax(logits.float(), dim=-1).gather(
        1, targets.unsqueeze(1)
    )
    return token_log_probs.mean().item()


## Part 15 — Combine model, rhyme, and meter scores

The final score combines the QLoRA model likelihood with pronunciation-based rhyme, syllable consistency, and lexicon quality.

In [ ]:

"""
TRAINING PROCESS — PART 15: HYBRID CANDIDATE SCORING

Purpose:
    Rerank neural candidates using explicit poetry constraints.

Methodology:
    Combine:
        neural plausibility
        + rhyme reward
        - syllable-count penalty

    The weights can be tuned experimentally.
"""

def candidate_score(
    line,
    language_model_score,
    rhyme_target=None,
    target_syllables=TARGET_SYLLABLES,
    rhyme_weight=RHYME_WEIGHT,
    meter_weight=0.15,
    lexicon_penalty=LEXICON_PENALTY
):
    score = language_model_score

    # --------------------------------------------------------------
    # Coherence component (English-ness)
    # --------------------------------------------------------------
    # Penalize lines made of words the lexicon does not recognize,
    # i.e. invented or garbled spellings.
    score -= lexicon_penalty * unknown_word_fraction(line)

    # --------------------------------------------------------------
    # Rhyme component
    # --------------------------------------------------------------
    if rhyme_target is not None:
        target_end_word = last_word(rhyme_target)
        candidate_end_word = last_word(line)

        score += rhyme_weight * rhyme_score(
            target_end_word,
            candidate_end_word
        )

    # --------------------------------------------------------------
    # Meter / syllable component
    # --------------------------------------------------------------
    syllables = syllable_count_line(line)

    # Unknown words can make the estimate zero; in that case
    # avoid applying a misleading penalty.
    if syllables > 0:
        meter_error = abs(syllables - target_syllables)
        score -= meter_weight * meter_error

    return score


## Part 16 — Generate a complete diss sonnet

Stage 2 is trained mostly on couplets. For each new rhyme pair, the model samples an anchor from the trait prompt and then samples its partner after that anchor. Lines 3/1, 4/2, 7/5, 8/6, 11/9, 12/10, and 14/13 must rhyme. When no sampled line qualifies and `ALLOW_CORPUS_FALLBACK=True`, the generator may copy a trait-relevant diss-corpus line; copied lines are marked and counted.

With corpus fallback disabled, an incomplete attempt returns `None` and records the failed line and candidate counts in `generate_diss_poem.last_stats`. The example and metrics cells handle this as a failed attempt, rather than treating a partial poem as a complete sonnet.

With the default `METER_TOLERANCE = 0`, every accepted line must have exactly 10 **estimated** syllables. Pronunciation estimates and poetic contractions are imperfect; this check does not enforce the unstressed–stressed pattern of iambic pentameter. Strict meter can cause more incomplete model-only attempts.


In [ ]:
"""Build a 14-line ABAB CDCD EFEF GG diss sonnet, or return None."""
KNOWN_POSITIVE_TRAITS = {
    trait.strip().lower()
    for pair in diss_samples
    for trait in pair["positive_traits"].split(",")
    if trait.strip()
}
SONNET_RHYME_PARTNERS = {2: 0, 3: 1, 6: 4, 7: 5, 10: 8, 11: 9, 13: 12}


def match_positive_traits(user_query, max_traits=5):
    query_words = re.findall(r"[a-z0-9]+", user_query.lower())
    matches = []
    for trait in KNOWN_POSITIVE_TRAITS:
        trait_words = re.findall(r"[a-z0-9]+", trait)
        for start in range(len(query_words) - len(trait_words) + 1):
            if query_words[start:start + len(trait_words)] == trait_words:
                matches.append((start, -len(trait_words), trait))
                break
    return [trait for _, _, trait in sorted(matches)[:max_traits]]


def max_line_similarity(line, previous_lines):
    words = set(re.findall(r"[a-z0-9]+", line.lower()))
    if not words:
        return 0.0
    return max(
        (len(words & set(re.findall(r"[a-z0-9]+", old.lower()))) / len(words)
         for old in previous_lines),
        default=0.0,
    )


# Use attested end words from the two corpora for rhyme substitutions.
from collections import Counter, defaultdict
end_word_counts = Counter(
    word for sample in sonnet_samples for line in sample.splitlines()
    if (word := last_word(line))
)
end_word_counts.update(
    word for pair in diss_samples for line in pair["text"].splitlines()
    if (word := last_word(line))
)
rhyme_endings = defaultdict(Counter)
for word, frequency in end_word_counts.items():
    if word.isalpha() and len(word) > 1:
        for key in rhyme_keys(word):
            rhyme_endings[key][word] = frequency


def rhyme_options(target_word, limit=24):
    options = Counter()
    for key in rhyme_keys(target_word):
        options.update(rhyme_endings.get(key, {}))
    options.pop(target_word, None)
    return [word for word, _ in options.most_common(limit)]


def valid_line(line, target_syllables=TARGET_SYLLABLES):
    words = words_in_line(line)
    return (
        4 <= len(words) <= 16
        and bool(last_word(line))
        and abs(syllable_count_line(line) - target_syllables) <= METER_TOLERANCE
        and unknown_word_fraction(line) <= MAX_UNKNOWN_WORD_FRACTION
    )


# Retain usable diss lines as a last-resort source when sampling yields no line.
# The source pair's positive traits help keep the fallback on topic.
DISS_LINE_POOL = []
_seen_diss_lines = set()
for pair in diss_samples:
    traits = {trait.strip().lower() for trait in pair["positive_traits"].split(",")}
    for line in pair["text"].splitlines():
        line = line.strip()
        if line not in _seen_diss_lines and valid_line(line):
            DISS_LINE_POOL.append((line, traits))
            _seen_diss_lines.add(line)
print(f"Usable diss lines for generation fallback: {len(DISS_LINE_POOL):,}")
DISS_RHYME_LINES = defaultdict(list)
for line, _ in DISS_LINE_POOL:
    ending = last_word(line)
    syllables = syllable_count_line(line)
    for key in rhyme_keys(ending):
        DISS_RHYME_LINES[key].append((ending, syllables))


from functools import lru_cache
@lru_cache(maxsize=None)
def corpus_partner_meter_error(word, target_syllables):
    """Best available distinct rhyming line's distance from the meter target."""
    return min(
        (abs(syllables - target_syllables)
         for key in rhyme_keys(word)
         for ending, syllables in DISS_RHYME_LINES.get(key, [])
         if ending != word),
        default=float("inf"),
    )


def corpus_fallback_candidates(target_word, matched_traits, selected_lines,
                               target_syllables, limit=16):
    """Return relevant source lines when the model provides no usable line."""
    wanted = set(matched_traits)

    def rank(line, traits):
        return (
            abs(syllable_count_line(line) - target_syllables),
            -len(wanted & traits),
            max_line_similarity(line, selected_lines),
            unknown_word_fraction(line),
        )

    available = [(line, traits) for line, traits in DISS_LINE_POOL
                 if line not in selected_lines]
    if target_word:
        natural = [(line, traits) for line, traits in available
                   if rhyme_score(last_word(line), target_word)]
        if natural:
            natural.sort(key=lambda item: rank(*item))
            return [line for line, _ in natural[:limit]]

        # If no source line has the required ending, try attested rhyming
        # words on otherwise suitable source lines, then rerank with the LM.
        endings = rhyme_options(target_word)
        available.sort(key=lambda item: rank(*item))
        rewritten = []
        for line, traits in available[:24]:
            stem = re.sub(r"[A-Za-z]+[.!?,;:]*$", "", line).rstrip()
            for word in endings:
                revised = f"{stem} {word}"
                if valid_line(revised, target_syllables) and revised not in selected_lines:
                    rewritten.append((revised, traits))
        rewritten.sort(key=lambda item: rank(*item))
        return list(dict.fromkeys(line for line, _ in rewritten))[:limit]

    anchors = [(line, traits) for line, traits in available
               if corpus_partner_meter_error(last_word(line), target_syllables) <= 2]
    if not anchors:
        anchors = [(line, traits) for line, traits in available
                   if rhyme_options(last_word(line), limit=1)]
    anchors.sort(key=lambda item: rank(*item))
    return [line for line, _ in anchors[:limit]]


def score_line(model, prompt, line, selected_lines, rhyme_target, target_syllables):
    score = candidate_score(
        line, line_log_likelihood(model, prompt, line),
        rhyme_target=rhyme_target, target_syllables=target_syllables,
    )
    return score - REPEAT_PENALTY * max_line_similarity(line, selected_lines)


@torch.no_grad()
def generate_diss_poem(
    user_query, model, number_of_lines=14,
    candidates_per_line=CANDIDATES_PER_LINE,
    target_syllables=TARGET_SYLLABLES,
    temperature=GENERATION_TEMPERATURE,
):
    if number_of_lines != 14:
        raise ValueError("This notebook generates 14-line Shakespearean sonnets.")
    if target_syllables != TARGET_SYLLABLES:
        raise ValueError("Change TARGET_SYLLABLES in Part 2 and rerun Part 16 before generation.")
    user_query = clean_text(user_query, LOWERCASE).replace("\n", " ")
    if not user_query:
        raise ValueError("Enter a positive self-description.")
    matched = match_positive_traits(user_query)
    trait_prompt = ", ".join(matched) if matched else user_query
    print("Subject traits:", trait_prompt)

    selected_lines = []
    fallback_count = 0
    for index in range(14):
        target = selected_lines[SONNET_RHYME_PARTNERS[index]] if index in SONNET_RHYME_PARTNERS else None
        target_word = last_word(target) if target else None
        # Stage 2 trains mostly on couplets. Start each rhyme anchor from the
        # trait prompt, then ask for its partner after the anchor line.
        prompt = (
            f"Subject traits: {trait_prompt}\nDiss lines:\n"
            + (f"{target}\n" if target else "")
        )
        sampled = []
        attempts = 0
        nonempty_samples = 0
        for _ in range(MAX_GENERATION_ROUNDS):
            for _ in range(candidates_per_line):
                attempts += 1
                line = generate_candidate_line(model, prompt, temperature)
                nonempty_samples += bool(line)
                if valid_line(line, target_syllables) and line not in sampled and line not in selected_lines:
                    sampled.append(line)
            if target_word:
                if any(rhyme_score(last_word(line), target_word) for line in sampled):
                    break
            elif sampled:
                break

        if target_word:
            candidates = [line for line in sampled if rhyme_score(last_word(line), target_word)]
            if not candidates:
                endings = rhyme_options(target_word)
                replacements = []
                for line in sampled:
                    stem = re.sub(r"[A-Za-z]+[.!?,;:]*$", "", line).rstrip()
                    for word in endings:
                        revised = f"{stem} {word}"
                        if valid_line(revised, target_syllables) and revised not in selected_lines:
                            replacements.append(revised)
                # Limit extra model scoring while preferring good meter and common endings.
                replacements.sort(key=lambda line: (
                    abs(syllable_count_line(line) - target_syllables),
                    -end_word_counts[last_word(line)],
                ))
                candidates = list(dict.fromkeys(replacements))[:16]
        else:
            # Every anchor needs a distinct attested rhyming word for its later partner.
            # Prefer anchors with an attested partner close to the meter target.
            candidates = [line for line in sampled
                          if corpus_partner_meter_error(last_word(line), target_syllables) <= 2]

        used_corpus_fallback = False
        if not candidates and ALLOW_CORPUS_FALLBACK:
            candidates = corpus_fallback_candidates(
                target_word, matched, selected_lines, target_syllables
            )
            used_corpus_fallback = bool(candidates)
        if not candidates:
            reason = (
                f"Stopped at line {index + 1}: {nonempty_samples}/{attempts} nonempty "
                f"samples, {len(sampled)} passed the line filter, and none met "
                "the required rhyme/meter constraints."
            )
            generate_diss_poem.last_stats = {
                "completed": False,
                "lines_generated": len(selected_lines),
                "model_lines": len(selected_lines) - fallback_count,
                "corpus_fallback_lines": fallback_count,
                "failed_line": index + 1,
                "sample_attempts": attempts,
                "nonempty_samples": nonempty_samples,
                "valid_samples": len(sampled),
                "reason": reason,
            }
            print(reason)
            return None
        best = max(candidates, key=lambda line: score_line(
            model, prompt, line, selected_lines, target, target_syllables
        ))
        selected_lines.append(best)
        source_note = " (corpus fallback)" if used_corpus_fallback else ""
        fallback_count += int(used_corpus_fallback)
        print(f"{index + 1:2d}: {best} [{syllable_count_line(best)} syllables]{source_note}")

    generate_diss_poem.last_stats = {
        "completed": True,
        "lines_generated": 14,
        "model_lines": 14 - fallback_count,
        "corpus_fallback_lines": fallback_count,
    }
    print(f"Model lines: {14 - fallback_count}/14; corpus fallback: {fallback_count}/14")
    assert len(selected_lines) == 14
    assert all(rhyme_score(last_word(selected_lines[i]), last_word(selected_lines[j]))
               for i, j in SONNET_RHYME_PARTNERS.items())
    return "\n\n".join(
        "\n".join(selected_lines[start:end])
        for start, end in ((0, 4), (4, 8), (8, 12), (12, 14))
    )



## Part 17 — Generate an example poem

### What this block does
Runs the complete trained system on an example positive self-description.

### Methodology
Change `user_query` to the user's positive self-description. Known traits from `positive_traits` produce a targeted poem; a query with no lexical trait match produces a generic diss poem. For a formal project evaluation, use the **same fixed set of prompts** across all model variants so comparisons remain fair.


In [ ]:
"""Generate an example; an incomplete model-only attempt is reported as a result."""
user_query = "I am confident and knowledgeable"
poem = generate_diss_poem(
    user_query=user_query,
    model=model,
    number_of_lines=14,
    candidates_per_line=CANDIDATES_PER_LINE,
    target_syllables=TARGET_SYLLABLES,
    temperature=GENERATION_TEMPERATURE,
)

if poem is None:
    print("\nNo complete model-origin sonnet. See the generation statistics above.")
    print("No generated_poem.txt was written for this attempt.")
else:
    print("\n" + "=" * 70)
    print("GENERATED POEM")
    print("=" * 70)
    print(poem)
    (OUTPUT_DIR / "generated_poem.txt").write_text(poem, encoding="utf-8")


# Evaluation

## Part 18 — Rhyme and meter metrics

Rhyme accuracy counts all seven required sonnet pairs; an unknown pronunciation counts as a miss. Syllables use dictionary pronunciations where available and a spelling heuristic for unknown words. Both metrics are estimates and should be paired with human evaluation.


In [ ]:
"""Evaluate all seven sonnet rhyme pairs and the syllable target."""
def evaluate_rhyme_accuracy(poem):
    lines = [line.strip() for line in poem.splitlines() if line.strip()]
    if len(lines) != 14:
        raise ValueError(f"Expected a 14-line sonnet, got {len(lines)} lines")
    details = []
    for index, partner in SONNET_RHYME_PARTNERS.items():
        word_a, word_b = last_word(lines[partner]), last_word(lines[index])
        details.append({
            "line_pair": f"{partner + 1}-{index + 1}",
            "word_a": word_a,
            "word_b": word_b,
            "rhymes": bool(rhyme_score(word_a, word_b)),
        })
    return sum(item["rhymes"] for item in details) / 7, details


def evaluate_syllable_error(poem, target_syllables=TARGET_SYLLABLES):
    lines = [line.strip() for line in poem.splitlines() if line.strip()]
    if len(lines) != 14:
        raise ValueError(f"Expected a 14-line sonnet, got {len(lines)} lines")
    details = []
    for index, line in enumerate(lines, start=1):
        count = syllable_count_line(line)
        details.append({
            "line": index, "syllables": count,
            "absolute_error": abs(count - target_syllables),
        })
    return sum(item["absolute_error"] for item in details) / 14, details


if poem is None:
    rhyme_accuracy = None
    meter_error = None
    exact_syllable_rate = None
    rhyme_details = []
    meter_details = []
    print("Poetry metrics skipped: no complete 14-line poem was generated.")
else:
    rhyme_accuracy, rhyme_details = evaluate_rhyme_accuracy(poem)
    meter_error, meter_details = evaluate_syllable_error(poem, TARGET_SYLLABLES)
    exact_syllable_rate = sum(
        item["syllables"] == TARGET_SYLLABLES for item in meter_details
    ) / 14
    print("Rhyme accuracy:", rhyme_accuracy)
    print(f"Exact {TARGET_SYLLABLES}-syllable line rate:", exact_syllable_rate)
    print("Mean absolute syllable error:", meter_error)
    print("Rhyme details:")
    print(pd.DataFrame(rhyme_details).to_string(index=False))
    print("Meter details:")
    print(pd.DataFrame(meter_details).to_string(index=False))


## Part 19 — Evaluate form and originality

Use the same held-out trait prompts for the base model, sonnet-stage adapter, and final diss-stage adapter. Report validation loss, completed-sonnet rate, rhyme accuracy, exact-10-syllable rate and syllable error **for completed poems**, plus the number of model-origin versus corpus-fallback lines. Rhyme accuracy with the constrained generator measures the constraint system, not the model alone. Set `ALLOW_CORPUS_FALLBACK = False` to measure how often the model completes a sonnet without copied corpus lines; a failed attempt is recorded, and poem-level metrics are skipped.

Human raters should separately judge whether each quatrain develops the same target, whether the closing couplet delivers a turn or punchline, whether lines sound like iambic pentameter, and whether copied dataset lines are acceptable. A lower syllable error alone does not establish meter or coherence.

For a stronger model, curate complete 14-line diss sonnets conditioned on traits. The current diss corpus is mostly couplets, so stage 2 does not directly teach full-sonnet argument or a closing turn. An instruction-tuned small model is another experiment, but switching models requires changing its prompt format and retraining the adapter; changing `BASE_MODEL_NAME` alone is not sufficient.



## Part 20 — Optional helper: save experiment metadata

### What this block does
Writes the main settings and final metrics to a JSON file.

### Methodology
Saving experiment metadata makes later report writing and model comparison easier, especially if you run several configurations with different hyperparameters.


In [ ]:
"""Save settings and final automatic metrics for this run."""
trainable_parameter_count = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
experiment_metadata = {
    "seed": SEED,
    "device": str(DEVICE),
    "base_model_name": BASE_MODEL_NAME,
    "sonnet_dataset_name": SONNET_DATASET_NAME,
    "quantization": "4-bit NF4 with double quantization",
    "qlora_rank": QLORA_R,
    "qlora_alpha": QLORA_ALPHA,
    "qlora_dropout": QLORA_DROPOUT,
    "sonnet_learning_rate": SONNET_LEARNING_RATE,
    "diss_learning_rate": DISS_LEARNING_RATE,
    "sonnet_validation_loss": sonnet_metrics["eval_loss"],
    "diss_validation_loss": diss_metrics["eval_loss"],
    "trainable_parameter_count": trainable_parameter_count,
    "target_syllables": TARGET_SYLLABLES,
    "meter_tolerance": METER_TOLERANCE,
    "candidates_per_line": CANDIDATES_PER_LINE,
    "pronunciation_source": pronunciation_source,
    "rhyme_accuracy": rhyme_accuracy,
    "mean_absolute_syllable_error": meter_error,
    "exact_syllable_rate": exact_syllable_rate,
    "generation_stats": getattr(generate_diss_poem, "last_stats", {}),
}
metadata_path = OUTPUT_DIR / "qlora_experiment_metadata.json"
with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(experiment_metadata, f, indent=2)
print("Saved experiment metadata to:", metadata_path)


## Part 21 — Package trained artifacts

The ZIP contains both LoRA adapters, their tokenizer, evaluation metadata, and the generated example. To use an adapter later, load `BASE_MODEL_NAME` and then apply the adapter with PEFT; the ZIP does not contain the pretrained base weights.


In [ ]:
"""Zip adapters, generated example, and experiment metadata."""
import shutil
archive_base = PROJECT_DIR.parent / "diss_poem_artifacts"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=PROJECT_DIR)
print("Created:", archive_path)
if IS_COLAB:
    from google.colab import files
    print("Download the archive if needed:")
    # files.download(archive_path)
else:
    print("Local archive:", archive_path)


# Summary

1. Load complete Shakespeare sonnets from one source and current raw diss rows.
2. Fine-tune LoRA adapters on sonnets, then on trait-conditioned diss examples, with sample-level validation for each stage.
3. Generate exactly 14 lines in three quatrains and a closing couplet. Phonetic rhyme and syllable checks guide line selection.
4. Report validation loss, rhyme accuracy, and mean syllable error; review coherence, tone, and meter by hand.

The diss corpus is mostly couplets, so full-sonnet quality is an empirical question. The evaluation cell exposes rhyme and syllable failures rather than assuming the model learned the form perfectly.
